In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
from collections import Counter


class FocalLoss(nn.Module):
    def __init__(self, gamma=2, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma * ce).mean()
        return loss

# -------------------------------------------------------
# Echo State Network (Fixed + Stable)
# -------------------------------------------------------
class SimpleESN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim,
                 spectral_radius=0.9, leaking_rate=1.0, ridge_param=1e-6, device="cpu"):
        super(SimpleESN, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.spectral_radius = spectral_radius
        self.leaking_rate = leaking_rate
        self.ridge_param = ridge_param
        self.device = device

        # Initialize fixed reservoir weights
        W_raw = torch.randn(hidden_dim, hidden_dim)
        eigvals = torch.linalg.eigvals(W_raw).abs()
        W_raw /= eigvals.max() / spectral_radius
        self.W = W_raw.to(device)

        # Input and bias weights
        self.W_in = (torch.randn(hidden_dim, input_dim) * 0.1).to(device)
        self.bias = (torch.randn(hidden_dim) * 0.01).to(device)

        # Output weights (trained via ridge regression)
        self.W_out = None

    def forward(self, X):
        """X: [batch, time, features]"""
        batch_size, timesteps, _ = X.shape
        h = torch.zeros(batch_size, self.hidden_dim, device=self.device)
        states = []

        for t in range(timesteps):
            u = X[:, t, :]
            pre_activation = (u @ self.W_in.T) + (h @ self.W.T) + self.bias
            h = (1 - self.leaking_rate) * h + self.leaking_rate * torch.tanh(pre_activation)
            states.append(h)

        states = torch.stack(states, dim=1)  # [batch, time, hidden]
        return states

    # def fit(self, X, y):
    #     """Ridge regression training (single global fit)."""
    #     states = self.forward(X)[:, -1, :]  # use last state per sequence

    #     # Convert one-hot → integer labels
    #     if y.ndim > 1:
    #         y = torch.argmax(y, dim=1)

    #     # One-hot encode for ridge regression
    #     y_onehot = F.one_hot(y, num_classes=self.output_dim).float()

    #     XTX = states.T @ states
    #     XTY = states.T @ y_onehot
    #     ridge = self.ridge_param * torch.eye(XTX.shape[0], device=self.device)

    #     self.W_out = torch.linalg.solve(XTX + ridge, XTY)  # [hidden, output]

    # def predict(self, X):
    #     with torch.no_grad():
    #         states = self.forward(X)[:, -1, :]
    #         y_pred = states @ self.W_out  # [batch, output_dim]
    #         return torch.softmax(y_pred, dim=1)

    def fit(self, X, y, class_weights=None):
        states = self.forward(X)[:, -1, :]  # [N, hidden]

        if y.ndim == 1:
            y_onehot = F.one_hot(y, num_classes=self.output_dim).float()
        else:
            y_onehot = y.float()

        if class_weights is not None:
            w = torch.tensor([class_weights[int(yi)] for yi in y], device=self.device, dtype=states.dtype)
            w = w.unsqueeze(1)  # [N,1]
            w /= w.mean()       # normalize weights
            # Weighted ridge regression without giant matrix
            X_weighted = states * w
            XTX = states.T @ X_weighted
            XTY = states.T @ (w * y_onehot)
        else:
            XTX = states.T @ states
            XTY = states.T @ y_onehot

        ridge = self.ridge_param * torch.eye(XTX.shape[0], device=self.device)
        self.W_out = torch.linalg.solve(XTX + ridge, XTY)

    def predict(self, X):
        """
        Predict class probabilities.
        X: [batch, time, features]
        Returns: [batch, num_classes] softmax probabilities
        """
        with torch.no_grad():
            states = self.forward(X)[:, -1, :]  # Get last timestep: [batch, hidden]
            logits = states @ self.W_out  # [batch, num_classes]
            probs = torch.softmax(logits, dim=1)
            return probs

# -------------------------------------------------------
# Data Preparation
# -------------------------------------------------------
def undersample_sequences(X_tensor, y_tensor, target_ratio=0.001):
    """
    Randomly remove majority class samples.
    target_ratio=1.0 means all classes will have same count as minority
    """
    y_np = y_tensor.numpy()
    classes, counts = np.unique(y_np, return_counts=True)
    min_count = counts.min()
    
    X_balanced = []
    y_balanced = []
    
    for cls in classes:
        mask = (y_np == cls)
        X_cls = X_tensor[mask]
        y_cls = y_tensor[mask]
        
        current_count = len(X_cls)
        target_count = int(min_count / target_ratio)
        
        if current_count > target_count:
            # Undersample
            indices = np.random.choice(current_count, target_count, replace=False)
            X_cls = X_cls[indices]
            y_cls = y_cls[indices]
        
        X_balanced.append(X_cls)
        y_balanced.append(y_cls)
    
    X_balanced = torch.cat(X_balanced, dim=0)
    y_balanced = torch.cat(y_balanced, dim=0)
    
    # Shuffle
    idx = torch.randperm(len(X_balanced))
    
    print(f"[INFO] Before undersampling: {len(X_tensor)} samples")
    print(f"[INFO] After undersampling: {len(X_balanced)} samples")
    print(f"[INFO] New distribution: {Counter(y_balanced.numpy())}")
    
    return X_balanced[idx], y_balanced[idx]


def data_prep(df, input_lst, output_lst, seq_len=50, target_ratio=1.0):
    input_scaler = StandardScaler()
    X_scaled = input_scaler.fit_transform(df[input_lst])

    y = df[output_lst].values
    y = np.argmax(y, axis=1)  # convert one-hot → integer class

    num_samples = len(X_scaled) // seq_len
    input_size = len(input_lst)

    X_seq = X_scaled[:num_samples * seq_len].reshape(num_samples, seq_len, input_size)
    y_seq = y[:num_samples * seq_len].reshape(num_samples, seq_len)
    y_seq_last = y_seq[:, -1]  # last timestep label

    X_tensor = torch.tensor(X_seq, dtype=torch.float32)
    y_tensor = torch.tensor(y_seq_last, dtype=torch.long)

    print(f"[INFO] Created {num_samples} sequences of length {seq_len}")
    print(f"[INFO] Input shape: {X_tensor.shape}, Target shape: {y_tensor.shape}")

    X_tensor, y_tensor = undersample_sequences(X_tensor, y_tensor, target_ratio=target_ratio)

    return X_tensor, y_tensor, input_scaler


# -------------------------------------------------------
# Training Function
# -------------------------------------------------------
def train_esn(X_tensor, y_tensor, input_size, output_size, hidden_dim=300,
              epochs=1, model_name="esn_model.pt", recall_boost=1.0):

    device = torch.device("cpu")
    model = SimpleESN(input_size, hidden_dim, output_size, device=device).to(device)

    # Move to device
    X_tensor, y_tensor = X_tensor.to(device), y_tensor.to(device)

    # Split train/test
    n = len(X_tensor)
    idx = int(0.8 * n)
    X_train, y_train = X_tensor[:idx], y_tensor[:idx]
    X_test, y_test = X_tensor[idx:], y_tensor[idx:]

    # ESN training (ridge regression)
    print("[INFO] Fitting ESN readout weights...")

    # -----------------------------
    # Compute inverse-frequency class weights
    # -----------------------------
    y_np = y_tensor.numpy().astype(int).flatten()
    counts = np.bincount(y_np)
    class_weights = {i: (1.0 / (c * recall_boost)) for i, c in enumerate(counts)}
    print("[INFO] Class distribution:", counts)
    print("[INFO] Computed weights:", class_weights)


    model.fit(X_train, y_train, class_weights=class_weights)

    # Evaluate
    preds = model.predict(X_test)
    y_pred = torch.argmax(preds, dim=1).cpu().numpy()
    y_true = y_test.cpu().numpy()

    print("\n[RESULTS]")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=3))

    torch.save(model.state_dict(), model_name)
    print(f"[INFO] Model saved as {model_name}")

    return model

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import numpy as np


def train_logistic_baseline(X_tensor, y_tensor, output_lst, class_weights):
    """
    Simple logistic regression baseline for comparison.
    Flattens sequences into feature vectors.
    """
    # Flatten sequences: [N, seq_len, features] -> [N, seq_len * features]
    X_flat = X_tensor.reshape(len(X_tensor), -1).numpy()
    y = y_tensor.numpy()
    
    # Train/test split
    n = len(X_flat)
    idx = int(0.8 * n)
    X_train, X_test = X_flat[:idx], X_flat[idx:]
    y_train, y_test = y[:idx], y[idx:]
    
    # Train logistic regression
    print("[INFO] Training Logistic Regression...")
    model = LogisticRegression(max_iter=1000, class_weight=class_weights, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test)
    
    print("\n[LOGISTIC REGRESSION RESULTS]")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=output_lst, digits=3))
    
    return model


In [50]:
# df = pd.read_feather('../data/aursad/aursad_training.feather')
input_lst = ['q0','q1','q2','q3','q4','q5', 'x', 'y', 'z',
             'Current0','Current1','Current2','Current3','Current4','Current5',
             'Speed0','Speed1','Speed2','Speed3','Speed4','Speed5',
             'Temperature0','Temperature1','Temperature2','Temperature3','Temperature4','Temperature5']
output_lst = ['Normal operation', 'Screw Loosening', 'Damaged screw', 'Extra assembly component', 'Missing screw']

X_tensor, y_tensor, input_scaler = data_prep(df, input_lst, output_lst, seq_len=50, target_ratio=0.1)


[INFO] Created 124981 sequences of length 50
[INFO] Input shape: torch.Size([124981, 50, 27]), Target shape: torch.Size([124981])
[INFO] Before undersampling: 124981 samples
[INFO] After undersampling: 120460 samples
[INFO] New distribution: Counter({np.int64(1): 57500, np.int64(0): 44209, np.int64(2): 7109, np.int64(4): 5892, np.int64(3): 5750})


In [51]:
class_dict = Counter(y_tensor.numpy())

model_lr = train_logistic_baseline(X_tensor, y_tensor, output_lst, class_weights=class_dict)

[INFO] Training Logistic Regression...

[LOGISTIC REGRESSION RESULTS]
Accuracy: 0.5058940727212352
                          precision    recall  f1-score   support

        Normal operation      0.502     0.221     0.307      8832
         Screw Loosening      0.507     0.881     0.643     11608
           Damaged screw      0.400     0.001     0.003      1398
Extra assembly component      0.500     0.004     0.007      1124
           Missing screw      0.857     0.005     0.011      1130

                accuracy                          0.506     24092
               macro avg      0.553     0.222     0.194     24092
            weighted avg      0.515     0.506     0.423     24092



/home/bentoaz/cmse830_fds/venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [53]:
import joblib
joblib.dump(model_lr, 'log_reg_balanced_ratio0.1.pkl')

['log_reg_balanced_ratio0.1.pkl']

In [52]:
model = train_esn(X_tensor, y_tensor, input_size=len(input_lst), output_size=len(output_lst), model_name='esn_corected', recall_boost=1.0)


[INFO] Fitting ESN readout weights...
[INFO] Class distribution: [44209 57500  7109  5750  5892]
[INFO] Computed weights: {0: np.float64(2.2619828541699653e-05), 1: np.float64(1.7391304347826085e-05), 2: np.float64(0.00014066676044450696), 3: np.float64(0.00017391304347826088), 4: np.float64(0.00016972165648336727)}

[RESULTS]
Accuracy: 0.26842935414245395
              precision    recall  f1-score   support

           0      0.493     0.226     0.310      8832
           1      0.653     0.215     0.323     11608
           2      0.138     0.436     0.210      1398
           3      0.116     0.610     0.195      1124
           4      0.116     0.604     0.194      1130

    accuracy                          0.268     24092
   macro avg      0.303     0.418     0.246     24092
weighted avg      0.514     0.268     0.300     24092

[INFO] Model saved as esn_corected


In [37]:
model = train_esn(X_tensor, y_tensor, input_size=len(input_lst), output_size=len(output_lst), model_name='esn_corected', recall_boost=5.0)


[INFO] Fitting ESN readout weights...
[INFO] Class distribution: [44209 57500  7109  5750  5892]
[INFO] Computed weights: {0: np.float64(4.523965708339931e-06), 1: np.float64(3.4782608695652175e-06), 2: np.float64(2.8133352088901393e-05), 3: np.float64(3.478260869565217e-05), 4: np.float64(3.394433129667346e-05)}

[RESULTS]
Accuracy: 0.2645691515855886
              precision    recall  f1-score   support

           0      0.480     0.208     0.290      8856
           1      0.624     0.221     0.327     11406
           2      0.130     0.453     0.202      1415
           3      0.116     0.527     0.190      1238
           4      0.127     0.605     0.210      1177

    accuracy                          0.265     24092
   macro avg      0.295     0.403     0.244     24092
weighted avg      0.492     0.265     0.293     24092

[INFO] Model saved as esn_corected


In [38]:
model = train_esn(X_tensor, y_tensor, input_size=len(input_lst), output_size=len(output_lst), model_name='esn_corected', recall_boost=10.0)


[INFO] Fitting ESN readout weights...
[INFO] Class distribution: [44209 57500  7109  5750  5892]
[INFO] Computed weights: {0: np.float64(2.2619828541699656e-06), 1: np.float64(1.7391304347826088e-06), 2: np.float64(1.4066676044450696e-05), 3: np.float64(1.7391304347826085e-05), 4: np.float64(1.697216564833673e-05)}

[RESULTS]
Accuracy: 0.2684708616968288
              precision    recall  f1-score   support

           0      0.483     0.216     0.299      8856
           1      0.626     0.221     0.327     11406
           2      0.133     0.435     0.203      1415
           3      0.120     0.561     0.198      1238
           4      0.128     0.614     0.211      1177

    accuracy                          0.268     24092
   macro avg      0.298     0.410     0.247     24092
weighted avg      0.494     0.268     0.297     24092

[INFO] Model saved as esn_corected


In [39]:
model = train_esn(X_tensor, y_tensor, input_size=len(input_lst), output_size=len(output_lst), model_name='esn_corected', recall_boost=1000.0)


[INFO] Fitting ESN readout weights...
[INFO] Class distribution: [44209 57500  7109  5750  5892]
[INFO] Computed weights: {0: np.float64(2.2619828541699654e-08), 1: np.float64(1.7391304347826087e-08), 2: np.float64(1.4066676044450696e-07), 3: np.float64(1.7391304347826088e-07), 4: np.float64(1.6972165648336728e-07)}

[RESULTS]
Accuracy: 0.2741158890918147
              precision    recall  f1-score   support

           0      0.483     0.222     0.304      8856
           1      0.625     0.228     0.334     11406
           2      0.139     0.446     0.212      1415
           3      0.120     0.556     0.197      1238
           4      0.130     0.613     0.214      1177

    accuracy                          0.274     24092
   macro avg      0.299     0.413     0.252     24092
weighted avg      0.494     0.274     0.303     24092

[INFO] Model saved as esn_corected


In [32]:
df

,sample_nr,time,target_q_0,target_q_1,target_q_2,target_q_3,target_q_4,target_q_5,target_qd_0,target_qd_1,...,output_bit_register_72,Normal operation,Damaged screw,Extra assembly component,Missing screw,Damaged thread samples,Screw Loosening,x,y,z
0,3431,0.000,0.083315,-1.084336,1.301481,-0.186469,-0.041835,-1.601735,0.0,0.0,...,False,False,False,False,False,False,True,0.0,0.0,0.0
1,3431,0.010,0.083315,-1.084336,1.301481,-0.186469,-0.041835,-1.601735,0.0,0.0,...,False,False,False,False,False,False,True,0.0,0.0,0.0
2,3431,0.020,0.083315,-1.084336,1.301481,-0.186469,-0.041835,-1.601735,0.0,0.0,...,False,False,False,False,False,False,True,0.0,0.0,0.0
3,3431,0.030,0.083315,-1.084336,1.301481,-0.186469,-0.041835,-1.601735,0.0,0.0,...,False,False,False,False,False,False,True,0.0,0.0,0.0
4,3431,0.040,0.083315,-1.084336,1.301481,-0.186469,-0.041835,-1.601735,0.0,0.0,...,False,False,False,False,False,False,True,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6249069,2704,23695.879,0.086743,-1.085944,1.304110,-0.174707,-0.037001,-1.605179,0.0,0.0,...,False,True,False,False,False,False,False,0.0,0.0,0.0
6249070,2704,23695.889,0.086743,-1.085944,1.304110,-0.174707,-0.037001,-1.605179,0.0,0.0,...,False,True,False,False,False,False,False,0.0,0.0,0.0
6249071,2704,23695.899,0.086743,-1.085944,1.304110,-0.174707,-0.037001,-1.605179,0.0,0.0,...,False,True,False,False,False,False,False,0.0,0.0,0.0
6249072,2704,23695.909,0.086743,-1.085944,1.304110,-0.174707,-0.037001,-1.605179,0.0,0.0,...,False,True,False,False,False,False,False,0.0,0.0,0.0
